# NVIDIA Developer Journey State Model
**Sequence-based journey state inference from activity histories**

Reads from `developer_project.duckdb`. Requires `dev_segments_v1` from `01_clustering_pipeline.ipynb`.

Input tables: `activity_final`, `contact_final`, `dev_segments_v1`

Output: `dev_journey_v1` written back to DuckDB

What this notebook does:
1. Builds ordered activity sequences per developer
2. Maps raw activities to journey stages (discover → learn → prototype → integrate → deploy → advocate)
3. Fits a Hidden Markov Model (HMM) to infer latent journey states
4. Assigns each developer a current journey state + transition probabilities
5. Identifies stall points and high-value transition paths
6. Combines journey state with cluster segment for the full `persona × journey-state` view

## 0. Imports & Config

In [ ]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from hmmlearn import hmm
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 100)

con = duckdb.connect('developer_project.duckdb')
RANDOM_STATE = 42

print('Connected. Tables available:')
display(con.execute('SHOW TABLES').fetchdf())

## 1. Load Activity Sequences

Pull the full ordered activity history per developer, sorted by `activity_date`.
We keep the raw `activity` label (already lowercased from cleaning) and the date.

In [ ]:
sequences_query = """
SELECT
    dev_contact AS developer_id,
    activity_date,
    activity,
    activity_score,
    activity_role,
    activity_attendance
FROM activity_final
WHERE dev_contact IS NOT NULL
  AND activity_date IS NOT NULL
ORDER BY dev_contact, activity_date
"""

seq_df = con.execute(sequences_query).fetchdf()
print(f'Activity rows loaded: {len(seq_df):,}')
print(f'Unique developers   : {seq_df["developer_id"].nunique():,}')
display(seq_df.head(5))

## 2. Map Activities to Journey Stages

Each raw activity type is mapped to one of 6 ordered journey stages:

| Stage | Code | Description | Activity types |
|---|---|---|---|
| Discover | 0 | First awareness and registration | webinars, event registrations, product specific comms |
| Learn | 1 | Structured training and content | dli training, on-demand views, conf sessions live |
| Prototype | 2 | Hands-on experimentation | devzone downloads, ngc downloads, brev, hackathon |
| Integrate | 3 | Active platform integration | hosted api, forum contributions, bugs filed |
| Deploy | 4 | Production-level usage | repeated api + forum, contests |
| Advocate | 5 | Community leadership | speaker/presenter events, user feedback, program membership |

In [ ]:
# Stage mapping — activity label (lowercased) → stage code
STAGE_MAP = {
    # Discover
    'webinars'                 : 0,
    'eventy registrations'     : 0,
    'product specific comms'   : 0,
    'conference'               : 0,
    'other events'             : 0,

    # Learn
    'dli training'             : 1,
    'on-demand views'          : 1,
    'conf sessions live'       : 1,

    # Prototype
    'devzone downloads'        : 2,
    'ngc downloads'            : 2,
    'brev'                     : 2,
    'hackathon'                : 2,

    # Integrate
    'hosted api'               : 3,
    'forum contributions'      : 3,
    'bugs filed'               : 3,

    # Deploy  (using contests as a proxy for production-level engagement)
    'contests'                 : 4,

    # Advocate
    'user feedback'            : 5,
    'dev program membership'   : 5,
    'program applications'     : 5,
}

STAGE_NAMES = {
    0: 'discover',
    1: 'learn',
    2: 'prototype',
    3: 'integrate',
    4: 'deploy',
    5: 'advocate'
}

N_STAGES = len(STAGE_NAMES)

# Map and drop unmapped activities (can review what falls through)
seq_df['stage'] = seq_df['activity'].map(STAGE_MAP)

unmapped = seq_df[seq_df['stage'].isna()]['activity'].value_counts()
print(f'Unmapped activity types ({len(unmapped)} unique):')
display(unmapped)

# For speaker/presenter roles — override to advocate regardless of activity type
speaker_mask = (
    seq_df['activity_role'].str.contains('speaker|presenter', case=False, na=False)
)
seq_df.loc[speaker_mask, 'stage'] = 5

seq_df = seq_df.dropna(subset=['stage'])
seq_df['stage'] = seq_df['stage'].astype(int)

print(f'\nRows after stage mapping: {len(seq_df):,}')
print('\nStage distribution:')
display(
    seq_df['stage'].value_counts().sort_index()
    .rename_axis('stage_code')
    .reset_index()
    .assign(stage_name=lambda x: x['stage_code'].map(STAGE_NAMES))
)

## 3. Build Sequences for HMM

Each developer becomes one sequence of stage codes ordered by date.
The HMM will learn latent states that explain these observed sequences.

In [ ]:
# Group into per-developer ordered sequences
dev_sequences = (
    seq_df
    .sort_values(['developer_id', 'activity_date'])
    .groupby('developer_id')['stage']
    .apply(list)
    .reset_index()
    .rename(columns={'stage': 'stage_sequence'})
)

dev_sequences['seq_length'] = dev_sequences['stage_sequence'].apply(len)

print(f'Developers with sequences: {len(dev_sequences):,}')
print('\nSequence length distribution:')
display(dev_sequences['seq_length'].describe())

# Filter to developers with at least 3 activity events
# (shorter sequences don't give HMM enough signal)
min_seq_length = 3
dev_sequences_filtered = dev_sequences[dev_sequences['seq_length'] >= min_seq_length].copy()

print(f'\nDevelopers with >= {min_seq_length} activities: {len(dev_sequences_filtered):,}')

## 4. Fit Hidden Markov Model

We fit a Categorical HMM where:
- **Observations** = journey stage codes (0–5)
- **Hidden states** = latent journey states that explain the observed stage sequences

The number of hidden states (`n_components`) controls coarseness. Start with 5–6, which maps roughly to the expected NVIDIA developer archetypes.

Note: `hmmlearn` expects sequences concatenated into a single array with `lengths` tracking where each sequence starts/ends.

In [ ]:
# Prepare input format for hmmlearn
all_sequences = dev_sequences_filtered['stage_sequence'].tolist()
lengths = [len(s) for s in all_sequences]

# Concatenate all sequences into one array, shape (total_obs, 1)
X_hmm = np.concatenate(all_sequences).reshape(-1, 1)

print(f'Total observations fed to HMM: {len(X_hmm):,}')
print(f'Number of sequences          : {len(lengths):,}')

In [ ]:
# ── HMM config ────────────────────────────────────────────────────────────────
# n_components: number of latent journey states to learn
#   5 maps loosely to: early-stage, learning, building, advanced, advocate
#   Increase to 7-8 for finer-grained states
# n_iter: EM iterations. 100 is sufficient for convergence in most cases.

N_HIDDEN_STATES = 5

model = hmm.CategoricalHMM(
    n_components=N_HIDDEN_STATES,
    n_iter=100,
    random_state=RANDOM_STATE,
    verbose=False
)

print(f'Fitting HMM with {N_HIDDEN_STATES} hidden states...')
model.fit(X_hmm, lengths)
print('HMM fit complete.')
print(f'Log-likelihood: {model.score(X_hmm, lengths):.2f}')

## 5. Decode Hidden States Per Developer

Use Viterbi decoding to find the most likely sequence of hidden states for each developer.
We then take the **final hidden state** as the developer's current journey state.

In [ ]:
journey_records = []
start_idx = 0

for i, (dev_id, seq) in enumerate(zip(
    dev_sequences_filtered['developer_id'],
    dev_sequences_filtered['stage_sequence']
)):
    length = len(seq)
    obs = np.array(seq).reshape(-1, 1)

    # Viterbi: most likely hidden state path
    log_prob, hidden_states = model.decode(obs, algorithm='viterbi')

    current_state     = hidden_states[-1]          # current (most recent) hidden state
    dominant_state    = Counter(hidden_states).most_common(1)[0][0]  # most frequent
    observed_max_stage = max(seq)                  # deepest observed journey stage

    journey_records.append({
        'developer_id'       : dev_id,
        'current_hidden_state': current_state,
        'dominant_hidden_state': dominant_state,
        'seq_length'         : length,
        'observed_max_stage' : observed_max_stage,
        'observed_max_stage_name': STAGE_NAMES[observed_max_stage],
        'log_prob'           : log_prob,
        'hidden_state_sequence': hidden_states.tolist()
    })

journey_df = pd.DataFrame(journey_records)

print(f'Journey states decoded for {len(journey_df):,} developers.')
print('\nCurrent hidden state distribution:')
display(journey_df['current_hidden_state'].value_counts().sort_index())

## 6. Profile Hidden States

Each hidden state is profiled by what observed stages it most commonly emits. This tells us what each latent state actually *means* in developer journey terms.

In [ ]:
# HMM emission matrix: rows = hidden states, cols = observed stages (0-5)
emission_matrix = pd.DataFrame(
    model.emissionprob_,
    columns=[STAGE_NAMES[i] for i in range(N_STAGES)],
    index=[f'hidden_state_{i}' for i in range(N_HIDDEN_STATES)]
)

print('Emission matrix (prob of observing each stage given hidden state):')
display(emission_matrix.round(3))

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(
    emission_matrix,
    annot=True, fmt='.2f', cmap='Blues',
    linewidths=0.5, ax=ax
)
ax.set_title('HMM Emission Probabilities\n(Which journey stages does each hidden state emit?)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Observed Journey Stage')
ax.set_ylabel('Hidden State')
plt.tight_layout()
plt.savefig('hmm_emission_matrix.png', dpi=150)
plt.show()

In [ ]:
# Transition matrix: prob of moving from hidden state i to hidden state j
transition_matrix = pd.DataFrame(
    model.transmat_,
    index=[f'from_state_{i}' for i in range(N_HIDDEN_STATES)],
    columns=[f'to_state_{i}' for i in range(N_HIDDEN_STATES)]
)

print('Transition matrix (prob of moving between hidden states):')
display(transition_matrix.round(3))

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    transition_matrix,
    annot=True, fmt='.2f', cmap='Greens',
    linewidths=0.5, ax=ax
)
ax.set_title('HMM Transition Probabilities\n(How likely is each state-to-state move?)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('hmm_transition_matrix.png', dpi=150)
plt.show()

## 7. Assign Human-Readable Journey State Labels

After reviewing the emission matrix above, assign descriptive names to each hidden state. The emission probabilities tell you what each state *means* — the state that mostly emits `learn` is an early-stage learner, the one that emits `integrate` and `deploy` is a builder, etc.

In [ ]:
# Update these after reviewing the emission matrix above
JOURNEY_STATE_LABELS = {
    0: 'relabel_me',   # e.g. 'early_discoverer'
    1: 'relabel_me',   # e.g. 'active_learner'
    2: 'relabel_me',   # e.g. 'prototyper'
    3: 'relabel_me',   # e.g. 'integrator'
    4: 'relabel_me',   # e.g. 'champion'
}

journey_df['journey_state_label'] = journey_df['current_hidden_state'].map(JOURNEY_STATE_LABELS)

print('Journey state distribution:')
display(journey_df['journey_state_label'].value_counts())

## 8. Stall Point Analysis

Which journey stages do developers most often stop at? These are the intervention opportunities.

In [ ]:
# Merge with activity dates to get last activity per developer
last_activity = (
    seq_df.groupby('developer_id')['activity_date'].max()
    .reset_index()
    .rename(columns={'activity_date': 'last_activity_date'})
)

journey_df = journey_df.merge(last_activity, on='developer_id', how='left')
journey_df['days_since_last_activity'] = (
    pd.Timestamp.today() - pd.to_datetime(journey_df['last_activity_date'])
).dt.days

# Stalled = no activity in 90+ days
STALL_THRESHOLD_DAYS = 90
journey_df['is_stalled'] = journey_df['days_since_last_activity'] > STALL_THRESHOLD_DAYS

stall_by_stage = (
    journey_df[journey_df['is_stalled']]
    .groupby('observed_max_stage_name')['developer_id']
    .count()
    .sort_values(ascending=False)
    .rename('stalled_developers')
)

print(f'Stalled developers (>{STALL_THRESHOLD_DAYS} days inactive): {journey_df["is_stalled"].sum():,}')
print('\nStall points by deepest reached stage:')
display(stall_by_stage)

fig, ax = plt.subplots(figsize=(9, 5))
stall_by_stage.plot(kind='bar', ax=ax, color='tomato', edgecolor='white')
ax.set_title(f'Stall Points — Developers Inactive >{STALL_THRESHOLD_DAYS} Days\nby Deepest Journey Stage Reached',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Deepest Journey Stage')
ax.set_ylabel('Stalled Developer Count')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('stall_points.png', dpi=150)
plt.show()

## 9. Combine Journey State with Cluster Segment

This creates the full `persona × journey-state` view referenced in the ChatGPT strategy doc.
Requires `dev_segments_v1` from the clustering notebook.

In [ ]:
# Check segments table exists
tables = con.execute('SHOW TABLES').fetchdf()['name'].tolist()
if 'dev_segments_v1' not in tables:
    print('WARNING: dev_segments_v1 not found. Run 01_clustering_pipeline.ipynb first.')
else:
    segments = con.execute(
        'SELECT developer_id, cluster, segment_label FROM dev_segments_v1'
    ).fetchdf()

    combined = journey_df.merge(segments, on='developer_id', how='left')

    print('Persona × Journey-State cross-tab:')
    crosstab = pd.crosstab(
        combined['segment_label'].fillna('no_cluster'),
        combined['journey_state_label'].fillna('no_journey_state')
    )
    display(crosstab)

    # Heatmap
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.heatmap(
        crosstab, annot=True, fmt='d', cmap='YlGnBu',
        linewidths=0.5, ax=ax
    )
    ax.set_title('Developer Count by Persona (cluster) × Journey State',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Journey State')
    ax.set_ylabel('Persona Segment')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig('persona_x_journey_state.png', dpi=150)
    plt.show()

## 10. Write Journey Results to DuckDB

In [ ]:
output_cols = [
    'developer_id',
    'current_hidden_state',
    'dominant_hidden_state',
    'journey_state_label',
    'observed_max_stage',
    'observed_max_stage_name',
    'seq_length',
    'is_stalled',
    'days_since_last_activity',
    'last_activity_date',
    'log_prob'
]

journey_out = journey_df[output_cols].copy()

con.execute('DROP TABLE IF EXISTS dev_journey_v1')
con.execute('CREATE TABLE dev_journey_v1 AS SELECT * FROM journey_out')

print('dev_journey_v1 written to DuckDB.')
display(con.execute(
    'SELECT journey_state_label, COUNT(*) AS n FROM dev_journey_v1 GROUP BY journey_state_label ORDER BY n DESC'
).fetchdf())

## 11. Summary & Next Steps

**What you now have in DuckDB:**
- `dev_segments_v1` — static persona clusters (from notebook 01)
- `dev_journey_v1` — dynamic journey state per developer

**Next steps:**
- Join `dev_segments_v1` and `dev_journey_v1` on `developer_id` for the full `persona × journey-state` view
- Feed stalled developers per segment into the uplift/next-best-action model
- Use `transition_matrix` to understand which states naturally progress and where the bottlenecks are
- Add `sdk_download_final` as a market context enrichment layer

In [ ]:
# Final join preview
if 'dev_segments_v1' in con.execute('SHOW TABLES').fetchdf()['name'].tolist():
    display(con.execute("""
        SELECT
            s.developer_id,
            s.segment_label,
            j.journey_state_label,
            j.observed_max_stage_name,
            j.is_stalled,
            j.days_since_last_activity,
            s.learn_score,
            s.build_score,
            s.community_score
        FROM dev_segments_v1 s
        LEFT JOIN dev_journey_v1 j ON s.developer_id = j.developer_id
        LIMIT 10
    """).fetchdf())